In [ ]:
import sys
from pathlib import Path

# Add src/ to Python path for local module imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from pydantic import BaseModel

import os

import constants

from prompting_utils import create_system_prompt, add_examples_to_user_prompt

from model_utils import from_series_to_basemodel

from tqdm import tqdm

from anthropic import Anthropic

from pathlib import Path

import json

# Preliminari

In [11]:
# Configurazione OpenAI
client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0

MODEL = constants.CLAUDE_OPUS_4_6
RESULTS_FILE_NAME = 'results_' + 'opus-4.6_MMR' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

train_data, validation_data, test_data = data['train'], data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    train_data[col] = train_data[col].round().astype("Int64")
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")

len(train_data) = 184
len(validation_data) = 66
len(test_data) = 65


## Load Embeddings

In [5]:
with open(base_dir / "data" / "embeddings" / constants.EMBEDDING_FILE_NAME_MISTRAL, "r") as f:
    embeddings = [json.loads(line) for line in f]

embeddings_train = {
    row['id']: np.array(row['embedding'])
    for row in embeddings if row['split'] == 'train'
}
embeddings_validation = {
    row['id']: np.array(row['embedding'])
    for row in embeddings if row['split'] == 'validation'
}
embeddings_test = {
    row['id']: np.array(row['embedding'])
    for row in embeddings if row['split'] == 'test'
}
embeddings_test_validation = embeddings_test | embeddings_validation

# Generazione

## Preliminari generazione

In [ ]:
def find_similar(query_embedding: np.ndarray, embeddings: dict[int, np.ndarray], k: int = 3) -> list[int]:
    """
    Ritorna id dal più simile al meno simile
    embeddings: train reports embeddings
    """
    ids = [id_ for id_ in embeddings]
    matrix = np.stack([embeddings[id_] for id_ in ids])
    scores = matrix @ query_embedding
    top_k = np.argsort(scores)[::-1][:k]
    
    return [ids[i] for i in top_k]


def return_list_of_annotated_reports(ids: list[int], data: pd.DataFrame) -> list[constants.AnnotatedRectalCancerReport]:
    result = []
    for id in ids:
        series = data[data[constants.ID_COMULM_NAME] == id].iloc[0]
        text = series[constants.REPORT_COLUMN_NAME]
        report_data = from_series_to_basemodel(series, constants.RectalCancerStagingData)
        result.append(constants.AnnotatedRectalCancerReport(report_text=text, report_data=report_data))
    return result


def mmr_select(query_embedding: np.ndarray,
               train_embeddings: dict[int, np.ndarray],
               k: int = 3,
               lambda_param: float = 0.5) -> list[int]:
    """
    Ritorna id dal più simile al meno simile. Gli id seleionati sono scelti tramite MMR (Maximum Marginal Relevance).
    MMR seleziona iterativamente l'esempio che massimizza il trade-off tra rilevanza rispetto alla query e diversità rispetto agli esempi già scelti.
    embeddings: train reports embeddings
    lambda_param: 1.0 = solo rilevanza (come cosine puro)
                  0.0 = solo diversità
                  0.5 = bilanciato (default consigliato)
    """    
    selected = []
    selected_embs = []
    for _ in range(k):
        best_score = -np.inf
        best_id = None
        best_emb = None

        for id_, emb in train_embeddings.items():
            if id_ in selected:
                continue
            
            # Rilevanza rispetto alla query
            relevance = query_embedding @ emb

            # Penalità: similarità massima con esempi già scelti
            if selected_embs:
                redundancy = max(emb @ s for s in selected_embs)
            else:
                redundancy = 0

            score = lambda_param * relevance - (1 - lambda_param) * redundancy

            if score > best_score:
                best_score = score
                best_id = id_
                best_emb = emb

        selected.append(best_id)
        selected_embs.append(best_emb)

    return selected

In [7]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             user_prompt: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.messages.parse(
        model=model,
        max_tokens=1024,
        system=system_prompt,
        temperature=temperature,
        messages=[
            {'role': 'user', 'content': user_prompt}
        ],
        output_format=output_format
    )
    return response

In [8]:
print(MODEL)

claude-opus-4-6


In [9]:
# Esempio
test_row = test_data.iloc[0]
test_text = test_row.report_text
test_id = test_row['id']
similar = mmr_select(embeddings_test[test_id], embeddings_train)
similar = similar[::-1]
ann_list = return_list_of_annotated_reports(similar, train_data)
user_prompt = add_examples_to_user_prompt(test_text, ann_list)
print(user_prompt)

if True:
    result = extract_data_from_report(MODEL, user_prompt)

## Esempi


### Esempio 1

**Referto:**
IN CORRISPONDENZA DEL RETTO ALTO , A CIRCA 10 CM DALLO SFINTERE ANALE INTERNO SI SEGNALA FORMAZIONE AGGETTANTE CHE OCCUPA PRESSOCCHE CIRCONFERENZIALMENTE IL LUME , CON ESTENSIONE CRANIO-CAUDALE DI CIRCA 5,7 CM , A PREVALENTE SVILUPPO ENDOLUMINALE CON FINI DIGITAZIONI EXTRAPARIETALI.
LUNGO I VASI EMORROIDARI SONO PRESENTI LINFONODI DISOMOGENEI, IN NUMERO SUPERIORE A TRE, SOSPETTI, I MAGGIORI CON ASSE CORTO DI CIRCA 8 MM .
CONCLUSIONI; STADIAZIONE RM : NEOPLASIA DEL RETTO ALTO , T3 N2"

**Output:**
{"morfologia":"solido_polipoide","ore_inizio":12,"ore_fine":12,"spessore_parietale":null,"estensione_cranio_caudale":57,"distanza_oai":100,"posizione":{"basso":"no","medio":"no","alto":"si","giunzione":"no"},"riflessione_peritoneale_anteriore":"cavallo","infiltrazione_tessuto_adiposo":"si_5mm","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"pavimento_pelvico":"no","altro":"no"},"coinvolgimento_riflessione

In [10]:
if False: # To see the full output
    pprint(result.model_dump())
if True:  # To see the string output text
    print(result.content[0].text)
if True:  # To see the parsed output as a pydantic BaseModel
    print(result.content[0].parsed_output)
if True:
    print(result.content[0].parsed_output.model_dump(mode='json'))

{"morfologia":"solido_polipoide","ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":null,"distanza_oai":0,"posizione":{"basso":"si","medio":"no","alto":"no","giunzione":"no"},"riflessione_peritoneale_anteriore":"sotto","infiltrazione_tessuto_adiposo":"no","infiltrazione_sfinteri":"si","infiltrazione_organi_extra":"si","infiltrazione_organi_dettagli":{"pavimento_pelvico":"si","altro":"no"},"coinvolgimento_riflessione_peritoneale":"no","coinvolgimento_fascia_mesorettale":"no","numero_linfonodi_non_conosciuto":"conosciuto","linfonodi_sospetti":1,"sedi_linfonodi":{"mesorettali":"no","rettali_superiori":"no","otturatori":"no","iliaci":"no","altro":"si"},"depositi_tumorali":"si","emvi":"+","stadio_T":"T4b","stadio_N":"N1","stadio_N1c":"si","mrf":"+","metastasi":"MX"}
morfologia='solido_polipoide' ore_inizio=None ore_fine=None spessore_parietale=None estensione_cranio_caudale=None distanza_oai=0 posizione=PosizioneFlags(basso='si', medio='no', alto='no', g

# Inference

In [11]:
print(MODEL)

claude-opus-4-6


In [12]:
print(MODEL)

df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
similars = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    query_id = row[constants.ID_COMULM_NAME]
    query_text = row[constants.REPORT_COLUMN_NAME]
    # Cerca id di referti simili
    similar = mmr_select(embeddings_test_validation[query_id], embeddings_train, lambda_param=0.5)
    similar = similar[::-1]
    ann_list = return_list_of_annotated_reports(similar, train_data)
    user_prompt = add_examples_to_user_prompt(query_text, ann_list)
    output = extract_data_from_report(model=MODEL, user_prompt=user_prompt)
    
    similars.append(similar)
    splits.append(row['split'])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(query_id)
    actual.append(real.model_dump(mode='json'))
   
    if output is None:
        predicted.append('no output')
    else:
        predicted.append(output.content[0].parsed_output.model_dump(mode='json'))

claude-opus-4-6


100%|██████████| 131/131 [28:05<00:00, 12.86s/it]


In [13]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split, similar in zip(ids, actual, predicted, splits, similars):
    results_dicts.append(
        {
            'model': f'MMR_{MODEL}',
            'split': split,
            'id': int(id),
            'example_ids': similar,
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')